DATA FUSION


In [ ]:
import os
from PIL import Image
import numpy as np

def tile_fixed_stride_specific_no_rotation(image_path, label_path_1d, label_path_visual, output_dir, tile_size=512):
    """
    Tiles a single image and its corresponding labels into fixed 512x512 patches based on specific start positions
    without rotating the image.
    
    Parameters:
    - image_path (str): Path to the input image.
    - label_path_1d (str): Path to the 1D label image.
    - label_path_visual (str): Path to the visual label image.
    - output_dir (str): Directory to save the output patches.
    - tile_size (int): Size of each tile (default is 512).
    """
    # Open images with consistent modes
    image = Image.open(image_path).convert('RGB')  # Ensure RGB mode
    label_1d = Image.open(label_path_1d).convert('L')  # Ensure grayscale mode
    label_visual = Image.open(label_path_visual).convert('RGB')  # Ensure RGB mode

    # Convert to numpy arrays
    image = np.array(image)
    label_1d = np.array(label_1d)
    label_visual = np.array(label_visual)
    
    img_height, img_width = image.shape[:2]
    
    print(f"Processing Image: {image_path}")
    print(f"Image shape: {image.shape}")  # Debug statement
    print(f"Label 1D shape: {label_1d.shape}")  # Debug statement
    print(f"Label visual shape: {label_visual.shape}")  # Debug statement

    # Ensure output directories exist
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels_1D'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)

    count = 0

    # Define specific start positions based on actual image size and tile size
    # For height (650 pixels): 0, 650 - 512 = 138
    # For width (1250 pixels): 0, 512, 1250 - 512 = 738
    if img_height != 650 or img_width != 1250:
        print(f"Warning: Expected image size 650x1250, but got {img_height}x{img_width}. Proceeding with specific tiling.")
    
    list_j = [0, img_height - tile_size]  # [0, 138]
    list_i = [0, 512, img_width - tile_size] if img_width > tile_size else [0]

    print(f"Start positions for height (j): {list_j}")
    print(f"Start positions for width (i): {list_i}")

    for j in list_j:
        for i in list_i:
            # Ensure that the starting positions are within the image bounds
            if j < 0 or i < 0:
                print(f"Skipping invalid start position: ({j}, {i})")
                continue

            # Crop the image and labels
            tile_image = image[j:j + tile_size, i:i + tile_size]
            tile_label_1d = label_1d[j:j + tile_size, i:i + tile_size]
            tile_label_visual = label_visual[j:j + tile_size, i:i + tile_size]

            # Verify that the cropped regions are of the correct size
            if (tile_image.shape[0] != tile_size or tile_image.shape[1] != tile_size):
                print(f"Skipping image tile at ({j}, {i}) due to incorrect size: {tile_image.shape}")
                continue
            if (tile_label_1d.shape[0] != tile_size or tile_label_1d.shape[1] != tile_size):
                print(f"Skipping label_1D tile at ({j}, {i}) due to incorrect size: {tile_label_1d.shape}")
                continue
            if (tile_label_visual.shape[0] != tile_size or tile_label_visual.shape[1] != tile_size):
                print(f"Skipping label_visual tile at ({j}, {i}) due to incorrect size: {tile_label_visual.shape}")
                continue

            # Save tiles
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            tile_image_name = f"{base_name}_{count}.png"
            tile_label_1d_name = f"{base_name}_{count}_label_1D.png"
            tile_label_visual_name = f"{base_name}_{count}_label.png"

            Image.fromarray(tile_image).save(os.path.join(output_dir, 'images', tile_image_name))
            Image.fromarray(tile_label_1d).save(os.path.join(output_dir, 'labels_1D', tile_label_1d_name))
            Image.fromarray(tile_label_visual).save(os.path.join(output_dir, 'labels', tile_label_visual_name))

            print(f"Saved patch {count}: Image({j}:{j+tile_size}, {i}:{i+tile_size})")
            count += 1

    print(f"Tiled {count} patches from {image_path}")

def process_subset1_images_specific_no_rotation(input_dir, output_dir, tile_size=512):
    """
    Processes all images in the input directory by tiling them into fixed patches without rotating the images.
    
    Parameters:
    - input_dir (str): Directory containing 'images', 'labels_1D', and 'labels' subdirectories.
    - output_dir (str): Directory to save the output patches.
    - tile_size (int): Size of each tile (default is 512).
    """
    images_dir = os.path.join(input_dir, 'images')
    labels_dir_1d = os.path.join(input_dir, 'labels_1D')
    labels_dir_visual = os.path.join(input_dir, 'labels')

    # List all image files (assuming .jpg and .png formats)
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Found {len(image_files)} images in {images_dir}")

    for image_file in image_files:
        image_path = os.path.join(images_dir, image_file)
        base_name = os.path.splitext(image_file)[0]

        # Corresponding label files
        label_file_1d = base_name + '.png'  # Adjust extension if necessary
        label_file_visual = base_name + '.png'  # Adjust extension if necessary

        label_path_1d = os.path.join(labels_dir_1d, label_file_1d)
        label_path_visual = os.path.join(labels_dir_visual, label_file_visual)

        if os.path.exists(label_path_1d) and os.path.exists(label_path_visual):
            tile_fixed_stride_specific_no_rotation(
                image_path=image_path,
                label_path_1d=label_path_1d,
                label_path_visual=label_path_visual,
                output_dir=output_dir,
                tile_size=tile_size
            )
        else:
            print(f"Label files not found for image {image_file}. Skipping.")

# Example usage:
if __name__ == "__main__":
    # Define input and output directories
    input_dir_subset1_train = r'G:/Other computers/Mi PC/krestininis/OilDataset/OilDataset/train'
    output_dir_patches_train = r'G:/Other computers/Mi PC/krestininis/OilDataset/OilDataset/train_512'

    # Define tile size
    tile_size = 512

    # Process images without rotation
    process_subset1_images_specific_no_rotation(
        input_dir=input_dir_subset1_train,
        output_dir=output_dir_patches_train,
        tile_size=tile_size
    )


In [10]:
from PIL import Image

# Path to your image
image_path = 'M:\Mi unidad\CIMA2023\Documentos2023\Proyectos\Proy1-ImagenesEspectrales\data\imagenes\OilDataset\\test\images\img_0001.jpg'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Show the image (optional)
image.show()

Image size: (1250, 650)
Image mode: RGB


STANDARIZE CLASSES LABEL_1D

In [47]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/filtered_patches/labels'
# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("label.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 19096


In [22]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/OilDatasetSOS/test/gt'

# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("_mask.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 839


In [27]:
from PIL import Image
import numpy as np

# Path to your image
image_path = 'H:\Derrame_Data\OilDataset\\test\labels_1D\img_0002.png'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Convert the image to a NumPy array
image_array = np.array(image)

# Print some pixel values
print("Pixel values:")
print(image_array)  # This will print out the pixel values in array format

# Optionally, print the shape of the image (to confirm the channels)
print(f"Image array shape: {image_array.shape}")


Image size: (1250, 650)
Image mode: L
Pixel values:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 4 4 4]
 [0 0 0 ... 4 4 4]
 [0 0 0 ... 4 4 4]]
Image array shape: (650, 1250)


In [ ]:
from PIL import Image
import numpy as np

# Path to your image
image_path = 'H:\Derrame_Data\OilDatasetPatchesFiltered\\train\\labels\\img_0001_11_label.png'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Convert the image to a NumPy array
image_array = np.array(image)

# Print some pixel values
print("Pixel values:")
print(image_array)  # This will print out the pixel values in array format

# Get unique RGB values and their counts
# Reshape the array to have each pixel's RGB as a row
pixels = image_array.reshape(-1, image_array.shape[-1])

# Count the unique rows (RGB values) and their occurrences
unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

# Display the results
for color, count in zip(unique_colors, counts):
    print(f"Color {color} occurs {count} times")

# Optionally, print the shape of the image (to confirm the channels)
print(f"Image array shape: {image_array.shape}")

In [30]:
def convert_sos_masks_to_labels_1d(input_dir, output_dir):
    """
    Converts RGB mask images in the SOS dataset to single-channel labels_1D format.

    Parameters:
    - input_dir: Directory containing the original mask images (e.g., 'train').
    - output_dir: Directory to save the converted labels (e.g., 'train_1D').
    """
    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(input_dir) if f.endswith('_mask.png') or f.endswith('_mask.jpg')]

    for mask_file in mask_files:
        mask_path = os.path.join(input_dir, mask_file)

        # Load the RGB mask image
        mask_rgb = Image.open(mask_path).convert('RGB')
        mask_array = np.array(mask_rgb)

        # Initialize the label array
        label_array = np.zeros((mask_array.shape[0], mask_array.shape[1]), dtype=np.uint8)

        # Create a boolean mask for white pixels (Oil Spill)
        oil_spill_mask = np.all(mask_array == [255, 255, 255], axis=-1)

        # Set Oil Spill pixels to 1
        label_array[oil_spill_mask] = 1  # Oil Spill

        # The remaining pixels are already set to 0 (Sea Surface)

        # Convert the label array to an image
        label_image = Image.fromarray(label_array, mode='L')

        # Save the label image
        label_file = mask_file.replace('_mask.png', '_label_1D.png').replace('_mask.jpg', '_label_1D.png')
        label_path = os.path.join(output_dir, label_file)
        label_image.save(label_path)

        print(f"Converted and saved: {label_path}")


In [ ]:
# Paths to the SOS dataset directories
sos_train_dir = 'H:/Derrame_Data/OilDatasetSOS/train'  # Replace with your actual path
sos_train_1d_dir = 'H:/Derrame_Data/OilDatasetSOS/train_1D'  # New directory for converted masks

# Convert the masks in the training set
convert_sos_masks_to_labels_1d(sos_train_dir, sos_train_1d_dir)


In [ ]:
# Paths to the SOS dataset directories
sos_train_dir = 'H:/Derrame_Data/OilDatasetSOS/test/gt'  # Replace with your actual path
sos_train_1d_dir = 'H:/Derrame_Data/OilDatasetSOS/test_1D'  # New directory for converted masks

# Convert the masks in the training set
convert_sos_masks_to_labels_1d(sos_train_dir, sos_train_1d_dir)


In [32]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/OilDatasetSOS/train_1D'

# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("_label_1D.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 3354


FILTRAR POR CANTIDAD DE % DE PIXELES

In [2]:
import os
import shutil
import random
from PIL import Image
import numpy as np

random.seed(322)  # Ensure reproducibility

def create_filtered_subset(
    input_dir,
    output_dir,
    krestininis_sample_size=50,
    classes_of_interest=[1, 2, 3],
    land_class=None,
    land_percentage_range=(0.25, 0.45),
):
    """
    Creates a filtered subset of patches based on specific criteria and selects a defined sample size.

    Parameters:
    - input_dir: Directory containing the patches (with subdirectories 'images', 'labels_1D', 'labels').
    - output_dir: Directory where the filtered subset will be stored.
    - krestininis_sample_size: The number of patches to select after filtering.
    - classes_of_interest: List of class labels to check for inclusion.
    - land_class: Class label for 'Land' (optional). If None, land percentage is not considered.
    - land_percentage_range: Tuple indicating the inclusive range of land percentage to include.
    """
    # Create output directories
    output_images_dir = os.path.join(output_dir, 'images')
    output_labels_1d_dir = os.path.join(output_dir, 'labels_1D')
    output_labels_dir = os.path.join(output_dir, 'labels')

    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_1d_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)

    # Define directories in input
    images_dir = os.path.join(input_dir, 'images')
    labels_1d_dir = os.path.join(input_dir, 'labels_1D')
    labels_dir = os.path.join(input_dir, 'labels')

    label_files = [f for f in os.listdir(labels_1d_dir) if f.endswith('.png') or f.endswith('.jpg')]

    # Filtered list
    filtered_files = []

    for label_file in label_files:
        label_path = os.path.join(labels_1d_dir, label_file)
        label_image = Image.open(label_path).convert('L')  # Open as grayscale
        label_array = np.array(label_image)  # Convert to NumPy array

        # Check if label contains any of the classes of interest
        contains_interest_class = np.isin(label_array, classes_of_interest).any()

        # Compute land percentage (if land_class is specified)
        land_percentage_criteria = True
        if land_class is not None:
            total_pixels = label_array.size
            land_pixels = np.sum(label_array == land_class)
            land_percentage = land_pixels / total_pixels
            land_percentage_criteria = land_percentage >= land_percentage_range[0] and land_percentage <= land_percentage_range[1]

        # Include if it matches the criteria
        if contains_interest_class or land_percentage_criteria:
            filtered_files.append(label_file)

    # Randomly select patches from the filtered list
    selected_files = random.sample(filtered_files, min(krestininis_sample_size, len(filtered_files)))

    # Copy selected files to the output directory
    for label_file in selected_files:
        # Copy the label_1D file
        label_path = os.path.join(labels_1d_dir, label_file)
        shutil.copy2(label_path, os.path.join(output_labels_1d_dir, label_file))

        # Corresponding image and visual label files
        base_name = label_file.replace('_label_1D.png', '').replace('_label_1D.jpg', '')
        image_file_png = base_name + '.png'
        image_file_jpg = base_name + '.jpg'
        label_visual_file_png = base_name + '_label.png'
        label_visual_file_jpg = base_name + '_label.jpg'

        # Check and copy the image file
        image_path = None
        if os.path.exists(os.path.join(images_dir, image_file_png)):
            image_file = image_file_png
            image_path = os.path.join(images_dir, image_file)
        elif os.path.exists(os.path.join(images_dir, image_file_jpg)):
            image_file = image_file_jpg
            image_path = os.path.join(images_dir, image_file_jpg)

        if image_path:
            shutil.copy2(image_path, os.path.join(output_images_dir, image_file))

        # Check and copy the visual label file
        label_visual_path = None
        if os.path.exists(os.path.join(labels_dir, label_visual_file_png)):
            label_visual_file = label_visual_file_png
            label_visual_path = os.path.join(labels_dir, label_visual_file)
        elif os.path.exists(os.path.join(labels_dir, label_visual_file_jpg)):
            label_visual_file = label_visual_file_jpg
            label_visual_path = os.path.join(labels_dir, label_visual_file_jpg)

        if label_visual_path:
            shutil.copy2(label_visual_path, os.path.join(output_labels_dir, label_visual_file))

    print(f"Selected and copied {len(selected_files)} patches based on filtering criteria.")

# Example usage:
input_dir = 'D:/Derrame_Data/OilDatasetPatches512/train'  # Replace with your actual path
output_dir = 'D:/Derrame_Data/filtered_subset_train512_gan1500'  # Replace with your desired output path

# If land filtering is not needed, set `land_class=None`
create_filtered_subset(
    input_dir=input_dir,
    output_dir=output_dir,
    krestininis_sample_size=1500,
    classes_of_interest=[1, 2, 3],  # Classes to include
    land_class=True,  # Disable land filtering
    land_percentage_range=(0.15, 0.5)  # Still used if land_class is provided
)


Selected and copied 1500 patches based on filtering criteria.


CREATE SUBSET DATA

In [52]:
import os
import json
import numpy as np
from skimage.io import imread

def write_dict_to_json(file_json, dict_data):
    """
    Writes a dictionary to a JSON file.
    """
    with open(file_json, "w", encoding="utf-8") as fh:
        json.dump(dict_data, fh, indent=4)
    return

def filter_images(file_list, dataset_type):
    """
    Filters the file list to include only image files based on dataset type.

    Parameters:
    - file_list: List of files in the directory.
    - dataset_type: 'sos' or 'krest' indicating the dataset being processed.

    Returns:
    - List of image files matching the criteria.
    """
    valid_extensions = ['.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif']
    if dataset_type == 'sos':
        # Include only files ending with '_sat.jpg'
        return [file for file in file_list
                if file.endswith('_sat.jpg') and
                os.path.splitext(file)[1].lower() in valid_extensions]
    elif dataset_type == 'krest':
        # Include all valid image files
        return [file for file in file_list
                if os.path.splitext(file)[1].lower() in valid_extensions]
    else:
        # If dataset_type is unknown, return an empty list
        return []

def compute_stats_for_datasets(sos_dirs, krest_dirs, file_json):
    """
    Computes the mean and std of images in the provided directories.

    Parameters:
    - sos_dirs: List of directories containing SOS images.
    - krest_dirs: List of directories containing Krestininis images.
    - file_json: Path to the JSON file where results will be saved.
    """
    all_means = []
    all_stds = []
    total_images = 0

    # Process SOS dataset
    for dir_images in sos_dirs:
        if not os.path.isdir(dir_images):
            print(f"Directory {dir_images} does not exist. Skipping.")
            continue

        image_files = sorted(filter_images(os.listdir(dir_images), dataset_type='sos'))
        num_images = len(image_files)
        total_images += num_images
        print(f"Processing {num_images} images in {dir_images} (SOS dataset)")

        for idx, image_file in enumerate(image_files):
            image_path = os.path.join(dir_images, image_file)
            image = imread(image_path)
            image = image / 255.0  # Normalize pixel values to [0, 1]

            if image.ndim == 3:
                mean_per_channel = np.mean(image, axis=(0, 1))
                std_per_channel = np.std(image, axis=(0, 1))
            elif image.ndim == 2:
                mean_per_channel = np.mean(image)
                std_per_channel = np.std(image)
                mean_per_channel = np.array([mean_per_channel])
                std_per_channel = np.array([std_per_channel])
            else:
                print(f"Skipping image {image_path}: unexpected number of dimensions ({image.ndim}).")
                continue

            all_means.append(mean_per_channel)
            all_stds.append(std_per_channel)

    # Process Krestininis dataset
    for dir_images in krest_dirs:
        if not os.path.isdir(dir_images):
            print(f"Directory {dir_images} does not exist. Skipping.")
            continue

        image_files = sorted(filter_images(os.listdir(dir_images), dataset_type='krest'))
        num_images = len(image_files)
        total_images += num_images
        print(f"Processing {num_images} images in {dir_images} (Krestininis dataset)")

        for idx, image_file in enumerate(image_files):
            image_path = os.path.join(dir_images, image_file)
            image = imread(image_path)
            image = image / 255.0  # Normalize pixel values to [0, 1]

            if image.ndim == 3:
                mean_per_channel = np.mean(image, axis=(0, 1))
                std_per_channel = np.std(image, axis=(0, 1))
            elif image.ndim == 2:
                mean_per_channel = np.mean(image)
                std_per_channel = np.std(image)
                mean_per_channel = np.array([mean_per_channel])
                std_per_channel = np.array([std_per_channel])
            else:
                print(f"Skipping image {image_path}: unexpected number of dimensions ({image.ndim}).")
                continue

            all_means.append(mean_per_channel)
            all_stds.append(std_per_channel)

    if total_images == 0:
        print("No images found in the provided directories.")
        return

    # Convert lists to numpy arrays
    all_means = np.array(all_means)
    all_stds = np.array(all_stds)

    # Compute overall mean and std across all images and channels
    mean_of_images = np.mean(all_means, axis=0)
    std_of_images = np.mean(all_stds, axis=0)

    # Display the results
    if mean_of_images.size == 3:
        print(f"Overall Mean per channel (R, G, B): {mean_of_images}")
        print(f"Overall Std per channel (R, G, B): {std_of_images}")
    else:
        print(f"Overall Mean: {mean_of_images[0]}")
        print(f"Overall Std: {std_of_images[0]}")

    # Prepare dictionary to save
    dict_stats = {
        "mean": mean_of_images.tolist(),
        "std": std_of_images.tolist()
    }

    write_dict_to_json(file_json, dict_stats)
    print(f"Image statistics saved in {file_json}")
    return

def main():
    # Directories for SOS dataset (images ending with '_sat.jpg')
    sos_dirs = [
        "H:\Derrame_Data\OilDatasetSOS\\train",
        "H:\Derrame_Data\OilDatasetSOS\\test",
    ]

    # Directories for Krestininis dataset
    krest_dirs = [
        "H:\Derrame_Data\\filtered_patches\\train\images",
        "H:\Derrame_Data\\filtered_patches\\test\images",
    ]

    file_json = "image_stats.json"

    compute_stats_for_datasets(sos_dirs, krest_dirs, file_json)

if __name__ == "__main__":
    main()


Processing 3354 images in H:\Derrame_Data\OilDatasetSOS\train (SOS dataset)
Processing 0 images in H:\Derrame_Data\OilDatasetSOS\test (SOS dataset)
Processing 19096 images in H:\Derrame_Data\filtered_patches\train\images (Krestininis dataset)
Processing 2028 images in H:\Derrame_Data\filtered_patches\test\images (Krestininis dataset)
Overall Mean per channel (R, G, B): [0.48539701 0.48539701 0.48539701]
Overall Std per channel (R, G, B): [0.17816202 0.17816202 0.17816202]
Image statistics saved in image_stats.json
